# Part D: Food-order stream with RabbitMQ
**Assignment: From Git to Gold**

This notebook runs the dispatcher and driver consumers for **Part D (D1, D2, D3)**.


## 1. Install & Start RabbitMQ in Colab

In [ ]:
!apt-get install -y -qq rabbitmq-server > /dev/null
!service rabbitmq-server start
!pip install -q pika


## 2. Verify RabbitMQ is healthy

In [ ]:
!rabbitmqctl status | head -n 15

## 3. Launch Drivers in Background
We start 4 drivers matching patterns from D1:
- **Ravi**: `bangalore.koramangala.*`
- **Arjun**: `bangalore.*.ride`
- **Meena**: `bangalore.*.food`
- **Control**: `bangalore.#`


In [ ]:
import subprocess, time

# Start background drivers and stream their outputs to log files
drivers = [
    ("Ravi", "bangalore.koramangala.*"),
    ("Arjun", "bangalore.*.ride"),
    ("Meena", "bangalore.*.food"),
    ("Control", "bangalore.#")
]

processes = []
for name, pattern in drivers:
    cmd = ["python", "stream/driver.py", name, pattern]
    log_file = open(f"{name.lower()}.log", "w")
    p = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
    processes.append((p, log_file, name))
    print(f"Started driver {name} with pattern '{pattern}' (PID {p.pid})")

time.sleep(2)  # Allow drivers to connect and bind queues


## 4. Run Dispatcher for 2 Minutes
Publishes orders every 2 seconds. Every 10th order intentionally has no `zone` to demonstrate D3 NACK behavior.


In [ ]:
import subprocess, time

print("Starting dispatcher for ~60 seconds (demonstrates 30 messages)...")
p_disp = subprocess.Popen(["python", "stream/dispatcher.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

start_time = time.time()
while time.time() - start_time < 60:
    line = p_disp.stdout.readline()
    if line:
        print(line, end="")
    time.sleep(0.1)

p_disp.terminate()
print("
Dispatcher run completed!")


## 5. Inspect Driver Logs (Proving D1 Routing & D3 NACK)
Check the logs of each driver to verify:
1. Routing pattern matches (D1)
2. `[ACCEPTED - ACK]` for valid messages (D3)
3. `[REJECTED - NACK]` for malformed messages (D3)


In [ ]:
# Stop drivers and close logs
for p, log_file, name in processes:
    p.terminate()
    log_file.close()

for name, pattern in drivers:
    print(f"\n{'='*25} LOG FOR {name} ({pattern}) {'='*25}")
    with open(f"{name.lower()}.log") as f:
        lines = f.readlines()
        print("".join(lines[:20]))  # print sample lines
